# Extension: Current Alcohol Use and Other Risk Behaviors

This extension examines whether the proportion of current alcohol use is different between students with other risk behaviors and students without other risk behaviors.

Other risk behavior is defined as either current cigarette use or physical fighting.

This is an exploratory extension using a two-proportion z-test.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

project_dir = Path("..")

raw_data_path = project_dir / "data" / "raw" / "YRBS_2007.csv"

figures_dir = project_dir / "outputs" / "figures"
tables_dir = project_dir / "outputs" / "tables"

figures_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(raw_data_path)

df.shape

In [ ]:
selected_vars = [
    "CurrentAlcoholUse",
    "CurrentCigaretteUse",
    "PhysicalFighting"
]

df_selected = df[selected_vars].copy()

for var in selected_vars:
    print(f"\n{var}")
    print(df_selected[var].value_counts(dropna=False).sort_index())

In [ ]:
df_clean = df_selected.dropna(subset=selected_vars).copy()

df_clean = df_clean[
    df_clean["CurrentAlcoholUse"].isin([1, 2, 3, 4, 5, 6, 7]) &
    df_clean["CurrentCigaretteUse"].isin([1, 2, 3, 4, 5, 6, 7]) &
    df_clean["PhysicalFighting"].isin([1, 2, 3, 4, 5, 6, 7, 8])
].copy()

# Response variable: current alcohol use
# 0 = did not currently use alcohol
# 1 = currently used alcohol
df_clean["current_alcohol_use"] = np.where(
    df_clean["CurrentAlcoholUse"] == 1,
    0,
    1
)

# Current cigarette use
# 0 = did not currently use cigarettes
# 1 = currently used cigarettes
df_clean["current_cigarette_use"] = np.where(
    df_clean["CurrentCigaretteUse"] == 1,
    0,
    1
)

# Physical fighting
# 0 = did not fight
# 1 = had at least one physical fight
df_clean["physical_fighting"] = np.where(
    df_clean["PhysicalFighting"] == 1,
    0,
    1
)

# Composite risk behavior group
# Other risk behavior = current cigarette use or physical fighting
df_clean["other_risk_behavior"] = np.where(
    (df_clean["current_cigarette_use"] == 1) |
    (df_clean["physical_fighting"] == 1),
    1,
    0
)

df_clean["risk_behavior_group"] = df_clean["other_risk_behavior"].map({
    0: "No other risk behavior",
    1: "Other risk behavior"
})

df_extension = df_clean[[
    "risk_behavior_group",
    "current_alcohol_use",
    "current_cigarette_use",
    "physical_fighting"
]].copy()

df_extension.head()

In [ ]:
summary_table = df_extension.groupby("risk_behavior_group").agg(
    sample_size=("current_alcohol_use", "count"),
    alcohol_users=("current_alcohol_use", "sum"),
    alcohol_use_proportion=("current_alcohol_use", "mean")
).reset_index()

summary_table["alcohol_use_percent"] = summary_table["alcohol_use_proportion"] * 100

summary_table

In [ ]:
group_summary = df_extension.groupby("risk_behavior_group").agg(
    sample_size=("current_alcohol_use", "count"),
    alcohol_users=("current_alcohol_use", "sum"),
    alcohol_use_proportion=("current_alcohol_use", "mean")
)

risk_success = group_summary.loc["Other risk behavior", "alcohol_users"]
risk_n = group_summary.loc["Other risk behavior", "sample_size"]
risk_prop = group_summary.loc["Other risk behavior", "alcohol_use_proportion"]

no_risk_success = group_summary.loc["No other risk behavior", "alcohol_users"]
no_risk_n = group_summary.loc["No other risk behavior", "sample_size"]
no_risk_prop = group_summary.loc["No other risk behavior", "alcohol_use_proportion"]

difference = risk_prop - no_risk_prop

count = np.array([risk_success, no_risk_success])
nobs = np.array([risk_n, no_risk_n])

z_stat, p_value = proportions_ztest(
    count=count,
    nobs=nobs,
    alternative="two-sided"
)

ci_low, ci_high = confint_proportions_2indep(
    count1=risk_success,
    nobs1=risk_n,
    count2=no_risk_success,
    nobs2=no_risk_n,
    method="wald"
)

print("Other risk behavior proportion:", risk_prop)
print("No other risk behavior proportion:", no_risk_prop)
print("Difference:", difference)
print("95% CI:", ci_low, "to", ci_high)
print("z statistic:", z_stat)
print("p-value:", p_value)

In [ ]:
extension_results = pd.DataFrame({
    "Statistic": [
        "Other risk behavior sample size",
        "Other risk behavior alcohol users",
        "Other risk behavior alcohol use proportion",
        "No other risk behavior sample size",
        "No other risk behavior alcohol users",
        "No other risk behavior alcohol use proportion",
        "Difference in proportions",
        "95% CI lower bound",
        "95% CI upper bound",
        "z statistic",
        "p-value"
    ],
    "Value": [
        risk_n,
        risk_success,
        risk_prop,
        no_risk_n,
        no_risk_success,
        no_risk_prop,
        difference,
        ci_low,
        ci_high,
        z_stat,
        p_value
    ]
})

summary_table.to_csv(
    tables_dir / "extension_summary_table_risk_behavior_alcohol.csv",
    index=False
)

extension_results.to_csv(
    tables_dir / "extension_inference_table_risk_behavior_alcohol.csv",
    index=False
)

extension_results

In [ ]:
processed_extension_path = project_dir / "data" / "processed" / "extension_cleaned_risk_behavior_alcohol.csv"

df_extension.to_csv(processed_extension_path, index=False)

print("Extension cleaned data saved to:", processed_extension_path)

In [ ]:
plt.figure(figsize=(7, 4))

plt.bar(
    summary_table["risk_behavior_group"],
    summary_table["alcohol_use_percent"]
)

plt.xlabel("Risk Behavior Group")
plt.ylabel("Current Alcohol Use (%)")
plt.title("Current Alcohol Use by Other Risk Behavior Group")
plt.ylim(0, 100)

plt.tight_layout()
plt.savefig(
    figures_dir / "extension_bar_chart_risk_behavior_alcohol.png",
    dpi=300
)
plt.show()

## Extension Interpretation

This extension compared current alcohol use between students with other risk behaviors and students without other risk behaviors.

Other risk behavior was defined as either current cigarette use or physical fighting.

The estimated proportion of current alcohol use was 0.6622 for students with other risk behaviors and 0.2825 for students without other risk behaviors.

The estimated difference in proportions was 0.3800, calculated as the other risk behavior group minus the no other risk behavior group. This means that students with other risk behaviors had about 38.00 percentage points higher current alcohol use than students without other risk behaviors.

The 95% confidence interval for the difference was from 0.3633 to 0.3967.

The two-proportion z-test gave a z statistic of 41.5471 and a p-value less than 0.001.

At the 0.05 significance level, we reject the null hypothesis.

This suggests that current alcohol use is statistically associated with other risk behaviors in this sample.

Because this dataset comes from an observational survey, this result should be interpreted as an association, not as a causal relationship.